In [1]:
import pandas as pd
import numpy as np

# Load dataset
df = pd.read_csv('most-popular-programming-languages-2004-2024.csv')

# Convert Month to datetime
df['Month'] = pd.to_datetime(df['Month'])

# Sort language columns alphabetically
languages = sorted([col for col in df.columns if col != 'Month'])
print("Languages:", languages)

# Student ID
student_id = 560
language_index = student_id % len(languages)

# Assigned language
assigned_language = languages[language_index]
print("Assigned Language:", assigned_language)

# Extract selected language data
lang_df = df[['Month', assigned_language]].copy()
lang_df.columns = ['Month', 'Popularity']

# Calculate growth rate
lang_df['Growth_Rate'] = lang_df['Popularity'].pct_change() * 100

# Rolling statistics
lang_df['Moving_Avg'] = lang_df['Popularity'].rolling(window=3).mean()
lang_df['Moving_STD'] = lang_df['Popularity'].rolling(window=3).std()

# Classify phases
conditions = [
    lang_df['Growth_Rate'] > 5,
    lang_df['Growth_Rate'] < -5
]

choices = ['Growth', 'Decline']

lang_df['Phase'] = np.select(conditions, choices, default='Stable')

# Statistical summary
summary = lang_df['Popularity'].describe()

# Overall growth
initial = lang_df['Popularity'].iloc[0]
final = lang_df['Popularity'].iloc[-1]
overall_growth = ((final - initial) / initial) * 100

# Phase counts
phase_counts = lang_df['Phase'].value_counts()

print(lang_df.head())
print("\nSummary:\n", summary)
print("\nPhase Counts:\n", phase_counts)
print("\nInitial:", initial)
print("Final:", final)
print("Overall Growth %:", overall_growth)


Languages: ['C# Worldwide(%)', 'Flutter Worldwide(%)', 'Java Worldwide(%)', 'JavaScript Worldwide(%)', 'Matlab Worldwide(%)', 'PhP Worldwide(%)', 'Python Worldwide(%)', 'React Worldwide(%)', 'Swift Worldwide(%)', 'TypeScript Worldwide(%)']
Assigned Language: C# Worldwide(%)
       Month  Popularity  Growth_Rate  Moving_Avg  Moving_STD    Phase
0 2004-01-01          76          NaN         NaN         NaN   Stable
1 2004-02-01          86    13.157895         NaN         NaN   Growth
2 2004-03-01          87     1.162791   83.000000    6.082763   Stable
3 2004-04-01          89     2.298851   87.333333    1.527525   Stable
4 2004-05-01          84    -5.617978   86.666667    2.516611  Decline

Summary:
 count    249.000000
mean      59.626506
std       20.330958
min       27.000000
25%       45.000000
50%       57.000000
75%       78.000000
max      100.000000
Name: Popularity, dtype: float64

Phase Counts:
 Phase
Stable     132
Decline     66
Growth      51
Name: count, dtype: int64

I

In [5]:
import pandas as pd
import numpy as np

df = pd.read_csv('most-popular-programming-languages-2004-2024.csv')
df['Month'] = pd.to_datetime(df['Month'])

# Find the column that contains C#
assigned_language = [col for col in df.columns if 'C#' in col][0]

print("Assigned Language Column:", assigned_language)

lang_df = df[['Month', assigned_language]].copy()
lang_df.columns = ['Month', 'Popularity']

lang_df['Growth_Rate'] = lang_df['Popularity'].pct_change() * 100
lang_df['Moving_Avg'] = lang_df['Popularity'].rolling(window=6).mean()
lang_df['Moving_STD'] = lang_df['Popularity'].rolling(window=6).std()

growth_clean = lang_df['Growth_Rate'].fillna(0)

mean_growth = growth_clean.mean()
std_growth = growth_clean.std()

conditions = [
    (growth_clean > 0) & (growth_clean < mean_growth),
    (growth_clean >= mean_growth),
    (abs(growth_clean) <= 1),
    (growth_clean < -std_growth)
]

choices = ['Introduction', 'Growth', 'Maturity', 'Decline']

lang_df['Lifecycle_Phase'] = np.select(
    conditions,
    choices,
    default='Stable'
)

print(lang_df.head())
print(lang_df['Lifecycle_Phase'].value_counts())


Assigned Language Column: C# Worldwide(%)
       Month  Popularity  Growth_Rate  Moving_Avg  Moving_STD Lifecycle_Phase
0 2004-01-01          76          NaN         NaN         NaN          Growth
1 2004-02-01          86    13.157895         NaN         NaN          Growth
2 2004-03-01          87     1.162791         NaN         NaN          Growth
3 2004-04-01          89     2.298851         NaN         NaN          Growth
4 2004-05-01          84    -5.617978         NaN         NaN          Stable
Lifecycle_Phase
Growth     136
Stable      72
Decline     41
Name: count, dtype: int64


In [7]:
import pandas as pd
import numpy as np

# Load dataset
df = pd.read_csv('most-popular-programming-languages-2004-2024.csv')

# Convert Month column to datetime
df['Month'] = pd.to_datetime(df['Month'])

# Print columns so we can verify names
print("Columns in dataset:")
print(df.columns.tolist())

# Automatically find C# and Flutter columns
lang1 = [col for col in df.columns if 'C#' in col][0]
lang2 = [col for col in df.columns if 'Flutter' in col][0]

print("Language 1:", lang1)
print("Language 2:", lang2)

# Create comparison dataframe
comp_df = df[['Month', lang1, lang2]].copy()
comp_df.columns = ['Month', 'CSharp', 'Flutter']

# Basic statistics
mean_a = comp_df['CSharp'].mean()
mean_b = comp_df['Flutter'].mean()

median_a = comp_df['CSharp'].median()
median_b = comp_df['Flutter'].median()

std_a = comp_df['CSharp'].std()
std_b = comp_df['Flutter'].std()

# Correlation
correlation = np.corrcoef(
    comp_df['CSharp'],
    comp_df['Flutter']
)[0, 1]

# Dominance ratio
count_a_greater = np.sum(comp_df['CSharp'] > comp_df['Flutter'])
total_months = len(comp_df)

dominance_ratio = (count_a_greater / total_months) * 100

# Relative Performance Index
rpi = mean_a / mean_b

# Cross-over points
cross_over = comp_df[
    (comp_df['CSharp'].shift(1) < comp_df['Flutter'].shift(1)) &
    (comp_df['CSharp'] > comp_df['Flutter'])
]

# Summary table
summary_df = pd.DataFrame({
    'Mean_CSharp': [mean_a],
    'Mean_Flutter': [mean_b],
    'Median_CSharp': [median_a],
    'Median_Flutter': [median_b],
    'Std_CSharp': [std_a],
    'Std_Flutter': [std_b],
    'Correlation': [correlation],
    'Dominance_Ratio (%)': [dominance_ratio],
    'RPI': [rpi]
})

# Output
print("\nComparative Data:")
print(comp_df.head())

print("\nSummary Statistics:")
print(summary_df)

print("\nCross-over Points:")
print(cross_over[['Month', 'CSharp', 'Flutter']])


Columns in dataset:
['Month', 'Python Worldwide(%)', 'JavaScript Worldwide(%)', 'Java Worldwide(%)', 'C# Worldwide(%)', 'PhP Worldwide(%)', 'Flutter Worldwide(%)', 'React Worldwide(%)', 'Swift Worldwide(%)', 'TypeScript Worldwide(%)', 'Matlab Worldwide(%)']
Language 1: C# Worldwide(%)
Language 2: Flutter Worldwide(%)

Comparative Data:
       Month  CSharp  Flutter
0 2004-01-01      76        6
1 2004-02-01      86        6
2 2004-03-01      87        5
3 2004-04-01      89        6
4 2004-05-01      84        6

Summary Statistics:
   Mean_CSharp  Mean_Flutter  Median_CSharp  Median_Flutter  Std_CSharp  \
0    59.626506      22.64257           57.0             8.0   20.330958   

   Std_Flutter  Correlation  Dominance_Ratio (%)       RPI  
0    28.368911    -0.676436            77.911647  2.633381  

Cross-over Points:
Empty DataFrame
Columns: [Month, CSharp, Flutter]
Index: []
